# ML-03: Framing My Lane as an ML Task

**Lane:** FlyRank content-performance warehouse — scoring content "opportunity" from
GSC/GA4 performance data (`FlyRank/internship-warehouse`).

**Rule followed:** core reasoning first, AI used only to stress-test and sharpen — not to author
the framing. This notebook went through several honest reframes before landing here; the earlier,
rejected framings are noted inline because they're part of the actual reasoning trail, not just
the final answer.


## 1) My lane as an ML task type

**Task type: Regression.**

The model predicts a continuous value — expected click-through rate (CTR) for a content item,
given its position, impression volume, and trend. The **opportunity score** is the residual:
`actual_CTR - predicted_CTR`. A large positive residual (predicted CTR well below actual — good)
or large negative residual (actual CTR well below what the item "should" get — an opportunity)
is the signal used to prioritize items, but ranking by that score is a downstream use of a
regression output, not a distinct ML task type in itself.

**Rejected earlier framing (kept here for honesty):** I first proposed "ranking" as the task type.
On inspection, `rank by impressions/clicks` turned out to be a plain sort — no ML needed. I also
considered a "fixability" classifier (will a flagged item's gap close later), but the dataset has
no ground-truth label for that, and manufacturing one from before/after windows was more label
engineering than this assignment's scope warranted. Regression on CTR residual is the version that
survived both checks.


## 2) Target or proxy

**Direct target, computed from the data itself — not an external label and not a proxy rule.**

`actual_CTR_90d = clicks_90d / impressions_90d`, both fields present in
`fact_content_query_90d`. The model predicts CTR from position/impressions/trend features; the
residual against this directly-computed actual CTR is the opportunity score. This avoids the
mistake from an earlier lane (biomarker framing) where the label risked being derived from the
same rule the model was supposed to beat — here, CTR is simply observed fact, not a rule-derived
proxy.


In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Attempt to load the real starter data ---
# FlyRank/internship-warehouse is gated on HuggingFace (requires accepting the dataset's
# terms + an HF token). A personal bucket copy also exists at Ollieboy/internship-warehouse-bucket.
# Try the bucket copy first since it doesn't require re-accepting the gate; fall back to the
# original repo path, then to a schema-matched synthetic scaffold if neither is reachable
# in this environment.
USE_REAL_DATA = False
candidate_paths = [
    ("Ollieboy/internship-warehouse-bucket", "fact_content_query_90d"),
    ("FlyRank/internship-warehouse", "fact_content_query_90d"),
]

for repo, config in candidate_paths:
    try:
        from datasets import load_dataset
        ds = load_dataset(repo, config, split="train")
        df = ds.to_pandas()
        USE_REAL_DATA = True
        print(f"Loaded real data from {repo} ({config}):", df.shape)
        break
    except Exception as e:
        print(f"Could not load {repo} ({e}).")

if not USE_REAL_DATA:
    print("Falling back to a schema-matched synthetic scaffold.")
    n = 2000
    avg_position_90d = np.abs(np.random.normal(20, 20, n)).clip(1, 100).round(2)
    # CTR generally falls off as position gets worse, plus noise -> synthetic but position-aware,
    # NOT a hand-set threshold rule like the earlier rejected scaffold.
    base_ctr = 0.35 / (1 + avg_position_90d / 5)
    noise = np.random.normal(0, 0.02, n)
    ctr = (base_ctr + noise).clip(0, 1)
    impressions_90d = np.random.lognormal(4, 1.5, n).astype(int).clip(1, None)
    clicks_90d = (impressions_90d * ctr).round().astype(int)

    impressions_prev30 = (impressions_90d * np.random.uniform(0.2, 0.4, n)).astype(int)
    impressions_last30 = (impressions_90d * np.random.uniform(0.2, 0.4, n)).astype(int)
    avg_position_prev30 = (avg_position_90d + np.random.normal(0, 3, n)).clip(1, 100).round(2)
    avg_position_last30 = (avg_position_90d + np.random.normal(0, 3, n)).clip(1, 100).round(2)

    df = pd.DataFrame({
        "client_hash_id": [f"c_{i%104}" for i in range(n)],
        "content_hash_id": [f"content_{i}" for i in range(n)],
        "impressions_90d": impressions_90d,
        "clicks_90d": clicks_90d,
        "avg_position_90d": avg_position_90d,
        "impressions_prev30": impressions_prev30,
        "impressions_last30": impressions_last30,
        "avg_position_prev30": avg_position_prev30,
        "avg_position_last30": avg_position_last30,
        "rare_impressions_share": np.random.uniform(0, 0.6, n).round(3),
    })

df.head()


Could not load Ollieboy/internship-warehouse-bucket (No module named 'datasets').
Could not load FlyRank/internship-warehouse (No module named 'datasets').
Falling back to a schema-matched synthetic scaffold.


,client_hash_id,content_hash_id,impressions_90d,clicks_90d,avg_position_90d,impressions_prev30,impressions_last30,avg_position_prev30,avg_position_last30,rare_impressions_share
0,c_0,content_0,14,1,29.93,3,3,29.64,19.80,0.511
1,c_1,content_1,52,4,17.23,12,18,16.06,15.88,0.033
2,c_2,content_2,56,2,32.95,14,19,32.30,35.24,0.089
3,c_3,content_3,110,3,50.46,25,27,51.46,49.29,0.346
4,c_4,content_4,7,0,15.32,2,2,14.81,13.92,0.444


## 3) Success metric

**Primary: MAE (mean absolute error)** on predicted CTR vs. actual CTR.

MAE was chosen over RMSE because it weighs every item's error equally rather than letting a small
number of extreme outliers (very high or very low CTR items, common in skewed traffic data) dominate
the score — the baseline needs to be fair across the bulk of ordinary content items, since that's
what most prioritization decisions will actually be made on. RMSE would be the better choice if
large misses were specifically dangerous to get wrong, which isn't the case for a prioritization
score used to guide human review.


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df["actual_ctr_90d"] = (df["clicks_90d"] / df["impressions_90d"]).clip(0, 1)

feature_cols = [
    "avg_position_90d",
    "impressions_90d",
    "avg_position_last30",
    "avg_position_prev30",
    "impressions_last30",
    "impressions_prev30",
]
X = df[feature_cols]
y = df["actual_ctr_90d"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("MAE: ", round(mae, 4), " <- primary metric")
print("RMSE:", round(rmse, 4))

if not USE_REAL_DATA:
    print("\nNOTE: trained on the synthetic scaffold (position-aware, not a threshold rule),")
    print("since the real warehouse wasn't reachable from this environment. Re-run against the")
    print("actual fact_content_query_90d table before treating these numbers as real results.")


MAE:  0.0403  <- primary metric
RMSE: 0.0542

NOTE: trained on the synthetic scaffold (position-aware, not a threshold rule),
since the real warehouse wasn't reachable from this environment. Re-run against the
actual fact_content_query_90d table before treating these numbers as real results.


## 4) Unit of analysis — as a real dataframe

**One row = one (client, content item) pair, aggregated over the fixed 90-day window.**
This matches `fact_content_query_90d`'s stated grain. I deliberately did **not** build this on
`fact_content_daily_performance` (one row per report date × client × content item) — checking
that table first showed the GA4 columns were almost entirely zero at daily grain and GSC clicks
were mostly zero too, because with ~520K content items and only 104 clients, most items don't get
traffic every single day. At daily grain, zero is the expected case, not a signal. The 90-day
aggregate (with `last30`/`prev30` sub-windows for trend) averages that noise out, which is why it's
the right table for this task.


## 5) Why ML beats a fixed rule here

**What I checked before claiming this:** I first proposed a plain rule — sort content items by
impressions descending, clicks ascending, to surface "opportunities." I tested this against my own
judgment and admitted it already surfaces what I care about — meaning a sort alone has no gap for
ML to fill. That framing was rejected.

I then proposed computing "expected CTR" via a **rule-based lookup**: bucket items into position
ranges (1-3, 4-10, 11-20, ...) and use the bucket's average CTR as the expected baseline. I checked
this honestly too, and it's a real, defensible option — simple, explainable, and a strong baseline
on its own. So the earlier claim that "you need a model just to get an expected-CTR curve" doesn't
hold; bucket-averaging does that adequately for a single variable.

**Where ML actually earns its place:** the moment you want the expected-CTR baseline to account for
**more than one variable at once** — position *and* impression volume *and* trend direction
(`last30` vs `prev30`) together — bucketing stops being practical. You'd need a separate bucket for
every combination of position range × impression range × trend direction, and most combinations
wouldn't have enough rows to produce a reliable average. A regression model handles several
continuous inputs jointly, smoothly, without hand-picking bucket boundaries for each one.

So the honest claim is narrower than my first instinct: ML isn't needed to compute *a* baseline —
it's needed to compute a baseline that accounts for **multiple signals together** rather than
position alone.


## 6) Self-check

- **Can I state task type, target, and metric in one sentence without hedging?**
  Regression predicting expected CTR for a (client, content item) pair from position, impression
  volume, and 90-day trend signals, evaluated by MAE against the actual observed CTR; the
  opportunity score is the residual.

- **If asked "why not just bucket by position and take the average?" — do I have an answer?**
  Yes: bucketing works fine for one variable, and I don't claim otherwise. The case for ML is
  specifically that I want position, impression volume, and trend combined, and bucketing across
  three continuous variables at once breaks down from sparse buckets — a regression model doesn't.

- **Anything here I'm not 100% sure I could defend live?**
  The synthetic fallback data (used only because this environment couldn't reach the gated
  HuggingFace dataset) is deliberately position-aware rather than rule-derived, so it doesn't repeat
  the earlier mistake of training on a label that IS the rule — but it's still synthetic, not real
  warehouse data. The MAE/RMSE numbers above are not meaningful until this notebook is re-run
  against the actual `fact_content_query_90d` table. I also don't yet know whether the FlyRank
  dataset's usage terms have been formally accepted for the bucket copy I'm using — worth confirming
  before this notebook or its outputs go anywhere public, since the terms restrict redistribution
  and require anonymized use only.
